In [ ]:
%reset -f
%load_ext autoreload
%autoreload 2

import sys
import os

sys.path.append(os.path.join(os.path.pardir, 'code'))
figdir = os.path.join(os.path.pardir, 'fig')
datadir = os.path.join(os.path.pardir, 'data')

import numpy as np
import pandas as pd
import scipy
import matplotlib.pyplot as plt
from matplotlib import cm, colors
from matplotlib.colors import ListedColormap
import pynumdiff
import pickle

import figurefirst as fifi
import figure_functions as ff

from eiso_fly_wind import eiso_fly_wind

from observability import empirical_observability_matrix, empirical_observability_matrix_sliding, num_jacobian
from eiso_brute import eiso_brute

from process_trajectory import Trajectory
from mpc_fly_wind import MpcFlyWind

pd.set_option('display.max_columns', None)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Load data

In [ ]:
# root = r"C:\Users\bcellini"  # UNR
root = r"C:\Users\BC"  # home

In [ ]:
rootpath = os.path.join(root, 'OneDrive - University of Nevada, Reno\Research\Data\wind_tunnel')
rootpath

'C:\\Users\\BC\\OneDrive - University of Nevada, Reno\\Research\\Data\\wind_tunnel'

In [ ]:
fname = 'laminar_orco_flash_qc_checked'
# fname = 'laminar_orco_sham_qc_checked'
# fname = 'laminar_wt_flash_qc_checked'
# fname = 'laminar_wt_sham_qc_checked'

In [ ]:
filepath = os.path.join(rootpath, fname + '.csv')
filepath

'C:\\Users\\BC\\OneDrive - University of Nevada, Reno\\Research\\Data\\wind_tunnel\\laminar_orco_flash_qc_checked.csv'

In [ ]:
data = pd.read_csv(filepath)
data.keys()

Index(['Unnamed: 0', 'Unnamed: 0_x', 'Unnamed: 0.1', 'obj_id', 'frame',
       'timestamp', 'x', 'y', 'z', 'xvel', 'yvel', 'zvel', 'P00', 'P01', 'P02',
       'P11', 'P12', 'P22', 'P33', 'P44', 'P55', 'millis', 'Flash_bool',
       'duration', 'last_flash', 'time_since_flash_millis',
       'time_since_flash_mins', 'obj_id_unique', 'orientation', 'time stamp',
       'heading', 'ang vel', 'theta smooth', 'theta dot smooth',
       'Unnamed: 0_y', 'amp', 'disp', 'score'],
      dtype='object')

### Number of trajectories

In [ ]:
# Find unique trajectories
trajectory_idx = data.obj_id_unique.unique()
n_trajectory = len(trajectory_idx)
print(n_trajectory, 'trajectories')

231 trajectories


### Seperate Trajectory types

In [ ]:
# Put each separate trajectory in list
control_list = []
full_list = []
half_list = []
for i in trajectory_idx:
    # Get trajectory
    traj = data[data.obj_id_unique == i]
    traj = traj.reset_index()
        
    # Normalize time & position
    traj.x = traj.x - traj.x.iloc[0]
    traj.y = traj.y - traj.y.iloc[0]
    traj.z = traj.z - traj.z.iloc[0]
    
    # Add to list
    exp_type = traj.duration.values[0]
    if exp_type == 100:
        full_list.append(traj)
    elif exp_type == 50:
        half_list.append(traj)
    elif exp_type == 0:
        control_list.append(traj)
        
print('Full:', len(full_list))
print('Half:', len(half_list))
print('Control:', len(control_list))

Full: 231
Half: 0
Control: 0


### Pick trajectory set to analyze

In [ ]:
# trajectory_list = full_list
# trajectory_list = control_list.copy()

trajectory_list = full_list + half_list + control_list
len(trajectory_list)

231

### Process all trajectories in set

In [ ]:
w0 = 0.4
zeta0 = 0.0

In [ ]:
DATA = []
for traj in trajectory_list:
    traj_data = Trajectory(traj, time_range=(-100, 1000), norm=True, fc=40, w=w0, zeta=zeta0)
    DATA.append(traj_data)

In [ ]:
len(DATA)

231

### Find controls with MPC

In [ ]:
# MPC paramters
n_horizon = 20
dt = np.round(DATA[0].dt, 5)

In [ ]:
# SI units
m = 0.25e-6 # [kg]
I = 5.2e-13  # [N*m*s^2] yaw mass moment of inertia: 10.1242/jeb.02369
# I = 4.971e-12  # [N*m*s^2] yaw mass moment of inertia: 10.1242/jeb.038778
C_phi = 27.36e-12  # [N*m*s] yaw damping: 10.1242/jeb.038778
C_para = m / 0.170  # [N*s/m] calculate using the mass and time constant reported in 10.1242/jeb.098665
# C_para = m / 0.050  # [N*s/m]
C_perp = C_para  # assume same as C_para

In [ ]:
# Convert to units of mg & mm to help with scaling for ODE solver
m = m * 1e6  # [mg]
I = I * 1e6 * (1e3)**2  # [mg*mm/s^2 * mm*s^2]
C_phi = C_phi * 1e6 * (1e3)**2  # [mg*mm/s^2 *m*s]
C_para = C_para * 1e6  # [mg/s]
C_perp = C_perp * 1e6  # [mg/s]

In [ ]:
print('m:', m, '\nI:', I, '\nC_phi:', C_phi, '\nC_para:', C_para, '\nC_perp:', C_perp)

m: 0.25 
I: 0.52 
C_phi: 27.36 
C_para: 1.4705882352941175 
C_perp: 1.4705882352941175


In [ ]:
x0 = {'I': I, 'm': m,
      'C_para': C_para, 'C_perp': C_perp, 'C_phi': C_phi,
      'd': 0.2,
      'km1': 1.0, 'km2': 0.0, 'km3': 1.0, 'km4': 1.0,
      'ks1': 1.0, 'ks2': 1.0, 'ks3': 0.0, 'ks4': 1.0, 'ks5': 0.0, 'ks6': 1.0, 'ks7': 0.0}

In [ ]:
MPC = []
for n, traj in enumerate(DATA):    
    # Set-points
    v_para = traj.traj['g_filt'].values
    v_perp = 0.0*np.ones_like(v_para)
    phi = traj.traj['phi_filt'].values
    w = w0*np.ones_like(v_para)
    zeta = zeta0*np.ones_like(v_para)

    # MPC
    mpc = MpcFlyWind(v_para, v_perp, phi, w, zeta, dt=dt, n_horizon=n_horizon, r_weight=1e-9, x0=x0)
    print(n, ':', mpc.error_metric)
    
    MPC.append(mpc)
    
print('Done')

0 : 0.003117147976561821
1 : 2.9484337826143505e-05
2 : 0.0005294175829091213


SystemError: <built-in function Function_call> returned a result with an error set

In [ ]:
MPC[1].plot_setpoint_tracking(size=4, lw=2)

### Observability matrix in sliding windows

In [ ]:
output_mode = ['phi', 'psi', 'gamma']

In [ ]:
OBSV = []
for n, mpc in enumerate(MPC):
    print(n, end=', ')
    
    x0 = mpc.x0
    fs = mpc.fs
    T = mpc.T

    # Controls from MPC
    r_para = mpc.u_mpc[:, 0]
    r_perp = mpc.u_mpc[:, 1]
    r_phi = mpc.u_mpc[:, 2]
    wdot = mpc.u_mpc[:, 3]
    zetadot = mpc.u_mpc[:, 4]
    
    # Simulate
    EISO = eiso_fly_wind(output_mode=output_mode, control_mode='open_loop', x0=x0, fs=fs, T=T, init_accel=True)
    EISO.simulate(r_para=r_para, r_perp=r_perp, r_phi=r_phi, wdot=wdot, zetadot=zetadot, init_accel=True)
    
    # Obervability in sliding windows
    O_sliding, O_time, O_index, window_data = empirical_observability_matrix_sliding(EISO.simulator, EISO.tsim, EISO.usim, EISO.xsim, eps=1e-5,
                                                                                     time_resolution=0.01, simulation_time=0.04)
    
    OBSV.append(O_sliding)

### EISO

In [ ]:
ej = 5  # zeta
beta = 1e-6

In [ ]:
CN = []
for n, O in enumerate(OBSV):
    print(n, end=', ')
    
    cn_state = []
    for w, o in enumerate(O):
        # print(w, end=', ')
        cn = EISO.eiso(O=o, ej_list=[ej], beta=1e-6, show_n_comb=False)
        cn_state.append(np.squeeze(cn.values))
        
    cn_state = np.stack(cn_state)
    CN.append(cn_state)

### Save

In [ ]:
keep_keys = ['v_para', 'v_perp', 'phi', 'phidot', 'w', 'zeta', 'x0', 'dt', 'fs',
             'n_horizon', 'n_points', 'T', 'tsim',
             'x_mpc', 'u_mpc', 'system', 'sim_data', 'sim_data_df',
             'v_para_error', 'v_perp_error', 'phi_error', 'error_metric']

In [ ]:
MPC_dict = []
for mpc in MPC:
    mpc_dict = mpc.__dict__
    mpc_dict_cut = {k: mpc_dict.get(k, None) for k in keep_keys}
    MPC_dict.append(mpc_dict_cut)

In [ ]:
pkdata = {'traj_data': DATA,
          'mpc': MPC_dict,
          'observability': OBSV,
          'O_time': O_time,
          'CN': CN,
          'beta': beta,
          'EISO': EISO
         }

In [ ]:
savepath = os.path.join(datadir, 'traj_data_' + fname + '.pk')
savepath